# Parte 1 · Noticias peruanas: recolección, limpieza y EDA

    **Medio elegido:** Opción A, La República. **Primera descarga comprobada:** 18 de octubre de 2026 (hora de Perú); la fecha y hora de cada nueva ejecución se registran en `datos/reporte_recoleccion.json` (UTC). **Fuentes base:** `https://larepublica.pe/rss/politica.xml`, `economia.xml`, `deportes.xml`, `ciencia.xml`; respaldo en sitemaps y páginas del medio. Como las fuentes oficiales del día solo dieron 53 noticias únicas, [Free News API](https://freenewsapi.ai/docs/search) sirve para descubrir URLs históricas. El recolector intenta verificar el HTML oficial y reporta cuándo solo obtuvo metadatos del índice. [Robots.txt](https://larepublica.pe/robots.txt) se consulta antes de cada artículo.

**Pregunta:** ¿Pueden convertirse los titulares peruanos en un clasificador de noticias? Este cuaderno produce el conjunto verificable y describe sus límites. Cada solicitud nueva espera 1,5 segundos y tiene un único reintento; los XML/HTML se guardan en `datos/raw/`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display
from recolectar import recolectar, SECCIONES, COLUMNAS

RAIZ = Path.cwd()
reporte = recolectar(objetivo=400)
print(json.dumps(reporte, ensure_ascii=False, indent=2))
assert reporte['cumple_minimo_350'], 'Faltan noticias reales: revisar RSS, sitemaps e índice, sin rellenar datos.'


## Limpieza y mapeo explícito

`recolectar.py` elimina titulares vacíos, URLs ajenas al medio, fechas ausentes, URLs y titulares repetidos. Reporta cada motivo; descarta cualquier sección fuera de las cuatro clases oficiales. La categoría es la sección editorial normalizada, por lo que puede haber casos ambiguos.


In [ ]:
df = pd.read_csv(RAIZ / 'datos/noticias.csv', encoding='utf-8-sig')
mapa = pd.DataFrame([{'seccion_original': s, 'etiqueta': e} for s, e in
                      {**SECCIONES, 'tecnologia': 'ciencia_tecnologia'}.items()])
display(mapa)
display(pd.DataFrame(reporte['descartes'].items(), columns=['motivo', 'noticias_descartadas']))
assert list(df.columns) == COLUMNAS
assert len(df) >= 350 and df['url'].is_unique and df['titular'].notna().all()
assert df['etiqueta'].isin(set(SECCIONES.values())).all()
print('Forma:', df.shape)
display(df.head(5))


## EDA · tres gráficos interactivos con conclusiones descriptivas generadas desde los datos


In [ ]:
conteos = df['etiqueta'].value_counts().rename_axis('categoría').reset_index(name='noticias')
mayor = conteos.iloc[0]
fig = px.bar(conteos, x='categoría', y='noticias', color='categoría',
             title=f"{mayor['categoría']} reúne más noticias ({mayor['noticias']}) en la muestra")
fig.show()


In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
assert df['fecha'].notna().all(), 'Hay fechas sin analizar: revisar la fuente.'
por_mes = df.assign(mes=df['fecha'].dt.to_period('M').astype(str)).groupby('mes').size().reset_index(name='noticias')
pico = por_mes.loc[por_mes['noticias'].idxmax()]
fig = px.bar(por_mes, x='mes', y='noticias',
             title=f"{pico['mes']} concentra más titulares ({pico['noticias']}) en el período descargado")
fig.show()


In [ ]:
df['caracteres'] = df['titular'].str.len()
mediana = int(df['caracteres'].median())
fig = px.histogram(df, x='caracteres', nbins=30,
                   title=f"La longitud mediana del titular es {mediana} caracteres")
fig.show()


## Comprobación de extremos y exportación


In [ ]:
print('Cinco titulares más largos:')
display(df.nlargest(5, 'caracteres')[['titular', 'caracteres', 'url']])
print('Cinco titulares más cortos:')
display(df.nsmallest(5, 'caracteres')[['titular', 'caracteres', 'url']])
df[COLUMNAS].to_csv(RAIZ / 'datos/noticias.csv', index=False, encoding='utf-8-sig')
print('Archivo exportado:', df[COLUMNAS].shape)
display(df[COLUMNAS].head(5))


## Verificación de enlaces

Se consultan las URLs por HTTP con pausas y caché de resultados. Una URL con dominio correcto no garantiza que el artículo siga publicado. Si hay fallos, revisar el archivo, corregir o excluir esas noticias y comprobar otra vez el mínimo de 350 filas.


In [ ]:
from validar_urls import main as validar_enlaces
validar_enlaces()
comprobacion = pd.read_csv(RAIZ / 'datos/validacion_urls.csv', encoding='utf-8-sig')
display(comprobacion['resultado'].value_counts().rename_axis('resultado').reset_index(name='enlaces'))
display(comprobacion[comprobacion['resultado'] != 'ok'].head(20))
print('Comprobar y documentar los fallos antes de entregar; una respuesta 403 puede indicar bloqueo del medio.')


Verificar la fecha, ejecutar todas las celdas y guardar el notebook con salidas antes de subirlo a GitHub.
